In [ ]:
import requests
class SEADParaibaAPI:
    def __init__(self):
        self.base_url = "https://api.dadosabertos.codata.pb.gov.br/api/v1/remuneracao"
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/json",
            "Content-Type": "application/json"
        })

    def listar_servidores(self, ano, mes, pagina=1):
        url = f"{self.base_url}/servidor"
        
        params = {
            "ano": str(ano),
            "mes": str(mes).zfill(2), 
            "page": pagina
        }
        
        print(f"Buscando URL EXATA: {url}")
        print(f"Parâmetros enviados: {params}...")
        
        response = self.session.get(url, params=params)
        
        if response.status_code != 200:
            print(f"Erro na API: {response.status_code}")
            print(response.text)
            return None

        return response.json()
    
    def consultar_remuneracao(self, ano, mes, nome_servidor=None):
        """Consulta os dados da folha de pagamento do servidor."""
        url = f"{self.base_url}/servidor/{ano}/{mes}"
        params = {}
        
        if nome_servidor:
            params["nome"] = nome_servidor
            
        response = self.session.get(url, params=params)
        response.raise_for_status()
        return response.json()

    def consultar_diarias(self, ano, nome_servidor=None):
        """Consulta as diárias emitidas para o servidor no ano vigente."""
        url = f"{self.base_url}/diarias"
        params = {"anoExercicio": ano}
        
        if nome_servidor:
            params["favorecido"] = nome_servidor
            
        response = self.session.get(url, params=params)
        response.raise_for_status()
        return response.json()

    def extrair_dossiê_servidor(self, ano, mes, nome_alvo):
        """Busca e consolida os dados de remuneração e diárias de um servidor."""
        print(f"Buscando dados de: {nome_alvo} ({mes}/{ano})...")
        
        dados_remuneracao = self.consultar_remuneracao(ano, mes, nome_alvo)
        
        dados_diarias = self.consultar_diarias(ano, nome_alvo)

        if not dados_remuneracao:
            print("Nenhum dado de remuneração encontrado.")
            return

        for registro in dados_remuneracao:
            if nome_alvo.upper() in str(registro.get("nome", "")).upper():
                
                total_diarias = sum(
                    float(d.get("valor", 0)) 
                    for d in dados_diarias 
                    if nome_alvo.upper() in str(d.get("nomeFavorecido", "")).upper()
                )
                
                dossie = {
                    "Órgão de lotação": registro.get("orgaoLotacao", "N/A"),
                    "Cargo": registro.get("cargo", "N/A"),
                    "Salário bruto": float(registro.get("remuneracaoBruta", 0)),
                    "Salário líquido": float(registro.get("remuneracaoLiquida", 0)),
                    "Gratificações": float(registro.get("gratificacoes", 0)),
                    "Vantagens (Pessoais/Indenizatórias)": float(registro.get("vantagens", 0)),
                    "Diárias recebidas (Acumulado do Ano)": total_diarias
                }
                
                self._imprimir_relatorio(nome_alvo, dossie)
                return

    def _imprimir_relatorio(self, nome, dossie):
        print(f"\n--- RELATÓRIO: {nome.upper()} ---")
        for chave, valor in dossie.items():
            if isinstance(valor, float):
                print(f"{chave}: R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))
            else:
                print(f"{chave}: {valor}")
        print("-----------------------------------")


if __name__ == "__main__":
    api = SEADParaibaAPI()
    
    dados = api.listar_servidores(ano=2025, mes=10, pagina=1)
    
    if dados:
        print("\n--- ESTRUTURA DO RETORNO DA API ---")
        print("Tipo do objeto:", type(dados))
        
        if isinstance(dados, dict):
            print("Chaves principais do JSON:", dados.keys())
            
            for chave, valor in dados.items():
                if isinstance(valor, list):
                    print(f"\nAchei a lista! Ela está dentro da chave: '{chave}'")
                    print(f"Quantidade de registros nesta página: {len(valor)}")
                    print("\nPrimeiro registro da lista para vermos os campos:")
                    print(valor[0])
                    break
            else:
                print("\nNenhuma lista encontrada diretamente nas chaves principais. Amostra do conteúdo:")
                print(str(dados)[:500])
                
        elif isinstance(dados, list):
            print("É uma lista direta! Primeiro item:")
            print(dados[0])

In [ ]:
import requests
import csv
import os

class SEADParaibaCSV:
    def __init__(self):
        self.base_url = "https://api.dadosabertos.codata.pb.gov.br/api/v1/remuneracao"
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/json",
            "Content-Type": "application/json"
        })

    def buscar_pagina(self, ano, mes, pagina):
        """Busca uma página específica da API."""
        url = f"{self.base_url}/servidor"
        params = {
            "ano": str(ano),
            "mes": str(mes).zfill(2),
            "page": pagina
        }
        
        response = self.session.get(url, params=params)
        
        if response.status_code == 200:
            return response.json()
        else:
            print(f"Erro ao buscar página {pagina}. Status: {response.status_code}")
            return None

    def localizar_lista(self, dados_json):
        """Descobre dinamicamente em qual chave a API guardou a lista de servidores."""
        if isinstance(dados_json, list):
            return dados_json
            
        if isinstance(dados_json, dict):
            for chave, valor in dados_json.items():
                if isinstance(valor, list):
                    return valor
        return []
    def exportar_para_csv(self, ano, mes, max_paginas=5):
            """
            Busca os dados e salva em um arquivo CSV contendo todas as colunas retornadas pela API.
            `max_paginas` define quantas páginas da folha baixar.
            """
            nome_arquivo = f"servidores_pb_{ano}_{str(mes).zfill(2)}.csv"
            
            print(f"Iniciando extração. Os dados serão salvos em: {nome_arquivo}\n")

            # Variáveis de controle para inicialização tardia do arquivo CSV (para descobrir os cabeçalhos dinamicamente)
            arquivo_csv = None
            escritor = None
            cabecalhos = []

            for pagina in range(1, max_paginas + 1):
                print(f"Baixando página {pagina}...")
                dados = self.buscar_pagina(ano, mes, pagina)
                
                if not dados:
                    break
                    
                lista_servidores = self.localizar_lista(dados)
                
                if not lista_servidores:
                    print("A lista de servidores veio vazia ou acabaram as páginas.")
                    break
                    
                import json
                # Processa cada servidor da página atual
                for servidor in lista_servidores:
                    print(f"Processando servidor: {json.dumps(servidor)}...")
                    
                    # Se for o primeiro registro encontrado, extraímos todas as chaves para montar os cabeçalhos dinamicamente
                    if not cabecalhos:
                        cabecalhos = list(servidor.keys())
                        
                        # Abre o arquivo CSV para escrita usando os cabeçalhos dinâmicos
                        arquivo_csv = open(nome_arquivo, mode="w", newline="", encoding="utf-8-sig")
                        escritor = csv.DictWriter(arquivo_csv, fieldnames=cabecalhos, delimiter=";")
                        escritor.writeheader()

                    # Escreve a linha mapeando todas as chaves diretamente do dicionário da API
                    escritor.writerow(servidor)

            # Garante o fechamento correto do arquivo caso ele tenha sido aberto
            if arquivo_csv:
                arquivo_csv.close()

            print(f"\nExtração concluída com sucesso! Arquivo salvo no caminho: {os.path.abspath(nome_arquivo)}")
# --- Execução do Código ---
if __name__ == "__main__":
    extrator = SEADParaibaCSV()
    
    # Extraindo dados de Outubro de 2025
    # Por padrão, configurei max_paginas=2 para teste rápido. 
    # Para baixar o estado inteiro, mude para um valor alto, ex: max_paginas=5000
    extrator.exportar_para_csv(ano=2025, mes=10, max_paginas=2)

In [ ]:
import pandas as pd
import plotly.express as px
import nbformat

# Carregar CSV
df = pd.read_csv(
    "folha.csv",
    sep=";",
    decimal=".",
    encoding="utf-8"
)

# Converter valores numéricos
colunas_numericas = [
    "vantagemFixa",
    "vantagemVariavel",
    "valorBruto",
    "valorPrevidenciario",
    "valorIr",
    "descontoObrigatorio",
    "valorDesconto",
    "valorLiquido"
]

for col in colunas_numericas:
    df[col] = pd.to_numeric(df[col], errors="coerce")


fig = px.histogram(
    df,
    x="valorLiquido",
    nbins=50,
    title="Distribuição dos Salários Líquidos"
)
fig.show()


top_cargos = (
    df.groupby("nomeCargo")["valorLiquido"]
    .mean()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)

fig = px.bar(
    top_cargos,
    x="valorLiquido",
    y="nomeCargo",
    orientation="h",
    title="Top 20 Cargos por Média Salarial"
)

fig.show()


top_unidades = (
    df.groupby("nomeUnidadeTrabalho")["valorLiquido"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)

fig = px.bar(
    top_unidades,
    x="valorLiquido",
    y="nomeUnidadeTrabalho",
    orientation="h",
    title="Top 20 Unidades por Gasto"
)

fig.show()


regime = (
    df.groupby("regimeContratual")
    .size()
    .reset_index(name="quantidade")
)

fig = px.pie(
    regime,
    names="regimeContratual",
    values="quantidade",
    title="Regime Contratual"
)

fig.show()


fig = px.box(
    df,
    x="sexo",
    y="valorLiquido",
    title="Distribuição Salarial por Sexo"
)

fig.show()


gastos_orgao = (
    df.groupby("orgaoLotacao")["valorLiquido"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)

fig = px.bar(
    gastos_orgao,
    x="valorLiquido",
    y="orgaoLotacao",
    orientation="h",
    title="Gasto Total por Órgão"
)

fig.show()


evolucao = (
    df.groupby("periodo")["valorLiquido"]
    .sum()
    .reset_index()
)

fig = px.line(
    evolucao,
    x="periodo",
    y="valorLiquido",
    markers=True,
    title="Evolução da Folha"
)

fig.show()


fig = px.box(
    df,
    y="valorLiquido",
    points="all",
    title="Outliers Salariais"
)

fig.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

class AnalisadorServidores:
    def __init__(self, df):
        self.df = df

    def analise_remuneracao(self):
        print("ANALISE DE REMUNERACAO")
        print("Remuneracao Media por Sexo:")
        remuneracao_sexo = self.df.groupby('sexo').agg({
            'valorBruto': ['mean', 'median', 'std'],
            'valorLiquido': ['mean', 'median'],
            'nomeServidor': 'count'
        }).round(2)
        print(remuneracao_sexo)

        print("Remuneracao Media por Tipo de Cargo (Top 10):")
        remuneracao_cargo = self.df.groupby('tipoCargo').agg({
            'valorBruto': ['mean', 'count']
        }).round(2).sort_values(('valorBruto', 'mean'), ascending=False).head(10)
        print(remuneracao_cargo)

        print("Remuneracao Media por Regime Contratual:")
        remuneracao_regime = self.df.groupby('regimeContratual').agg({
            'valorBruto': ['mean', 'median'],
            'nomeServidor': 'count'
        }).round(2)
        print(remuneracao_regime)

    def analise_descontos(self):
        print("ANALISE DE DESCONTOS E ENCARGOS")
        print("Descontos Totais:")
        print(f"  Desconto medio: R$ {self.df['valorDesconto'].mean():.2f}")
        print(f"  Desconto mediano: R$ {self.df['valorDesconto'].median():.2f}")
        print(f"  Desconto maximo: R$ {self.df['valorDesconto'].max():.2f}")
        print(f"  Percentual de desconto medio: {self.df['descontosPercentual'].mean():.2f}%")

        print("Composicao dos Descontos (valores medios):")
        descontos_composicao = self.df[['valorPrevidenciario', 'valorIr', 'descontoObrigatorio']].mean()
        total_desconto_medio = self.df['valorDesconto'].mean()
        for coluna in descontos_composicao.index:
            percentual = (descontos_composicao[coluna] / total_desconto_medio * 100) if total_desconto_medio > 0 else 0
            print(f"  {coluna}: R$ {descontos_composicao[coluna]:.2f} ({percentual:.1f}%)")

        print("Descontos por Situacao do Servidor:")
        descontos_situacao = self.df.groupby('situacaoServidor').agg({
            'valorDesconto': ['mean', 'median'],
            'valorPrevidenciario': 'mean',
            'valorIr': 'mean'
        }).round(2)
        print(descontos_situacao)

    def analise_anos_servico(self):
        print("ANALISE DE ANOS DE SERVICO")
        print("Estatisticas de Anos de Servico:")
        print(f"  Minimo: {self.df['anosServico'].min():.1f} anos")
        print(f"  Maximo: {self.df['anosServico'].max():.1f} anos")
        print(f"  Media: {self.df['anosServico'].mean():.1f} anos")
        print(f"  Mediana: {self.df['anosServico'].median():.1f} anos")

        self.df['categoriaAnosServico'] = pd.cut(
            self.df['anosServico'],
            bins=[0, 1, 5, 10, 20, float('inf')],
            labels=['< 1 ano', '1-5 anos', '5-10 anos', '10-20 anos', '> 20 anos']
        )

        print("Distribuicao por Categorias de Tempo de Servico:")
        dist_tempo = self.df['categoriaAnosServico'].value_counts().sort_index()
        for categoria, quantidade in dist_tempo.items():
            percentual = (quantidade / len(self.df)) * 100
            print(f"  {categoria}: {quantidade} ({percentual:.1f}%)")

        print("Correlacao: Anos de Servico vs Remuneracao:")
        correlacao = self.df['anosServico'].corr(self.df['valorBruto'])
        print(f"  Correlacao de Pearson: {correlacao:.3f}")

    def analise_distribuicao_orgaos(self):
        print("ANALISE DE DISTRIBUICAO POR ORGAOS")
        print("Top 10 Orgaos por Quantidade de Servidores:")
        orgaos = self.df['orgaoLotacao'].value_counts().head(10)
        for orgao, quantidade in orgaos.items():
            percentual = (quantidade / len(self.df)) * 100
            print(f"  {orgao}: {quantidade} ({percentual:.1f}%)")

        print("Top 10 Orgaos por Folha de Pagamento:")
        folha_por_orgao = self.df.groupby('orgaoLotacao')['valorBruto'].sum().sort_values(ascending=False).head(10)
        for orgao, folha in folha_por_orgao.items():
            print(f"  {orgao}: R$ {folha:,.2f}")

    def analise_escolaridade(self):
        print("ANALISE POR ESCOLARIDADE")
        print("Distribuicao por Escolaridade Minima:")
        escolaridade = self.df['escolaridadeMinimaCargo'].value_counts()
        for nivel, quantidade in escolaridade.items():
            percentual = (quantidade / len(self.df)) * 100
            print(f"  {nivel}: {quantidade} ({percentual:.1f}%)")

        print("Remuneracao Media por Escolaridade:")
        remuneracao_escolaridade = self.df.groupby('escolaridadeMinimaCargo').agg({
            'valorBruto': ['mean', 'median', 'count']
        }).round(2).sort_values(('valorBruto', 'mean'), ascending=False)
        print(remuneracao_escolaridade)

    def analise_pessoas_deficientes(self):
        print("ANALISE - PESSOAS COM DEFICIENCIA")
        total = len(self.df)
        com_deficiencia = (self.df['deficienteFisico'] == 'SIM').sum()
        sem_deficiencia = (self.df['deficienteFisico'] == 'NAO').sum()

        print(f"Total de Servidores: {total}")
        print(f"  Com deficiencia: {com_deficiencia} ({com_deficiencia/total*100:.2f}%)")
        print(f"  Sem deficiencia: {sem_deficiencia} ({sem_deficiencia/total*100:.2f}%)")

        if com_deficiencia > 0:
            print("Cargos de Pessoas com Deficiencia:")
            cargos_def = self.df[self.df['deficienteFisico'] == 'SIM']['nomeCargo'].value_counts().head(5)
            for cargo, quantidade in cargos_def.items():
                print(f"  {cargo}: {quantidade}")

    def analise_administracao_indireta(self):
        print("ANALISE - ADMINISTRACAO INDIRETA")
        adm_indireta = self.df[self.df['administracao'] == 'ADMINISTRACAO INDIRETA']

        print(f"Total de Servidores em Admin. Indireta: {len(adm_indireta)}")
        print(f"   Percentual do total: {len(adm_indireta)/len(self.df)*100:.1f}%")

        print("Distribuicao por Entidade:")
        entidades = adm_indireta['cnpjOrgao'].value_counts()
        for entidade, quantidade in entidades.items():
            print(f"  {entidade}: {quantidade}")

        print("Folha de Pagamento:")
        print(f"  Total: R$ {adm_indireta['valorBruto'].sum():,.2f}")
        print(f"  Media por servidor: R$ {adm_indireta['valorBruto'].mean():,.2f}")

    def gerar_relatorio_completo(self):
        self.analise_remuneracao()
        self.analise_descontos()
        self.analise_anos_servico()
        self.analise_distribuicao_orgaos()
        self.analise_escolaridade()
        self.analise_pessoas_deficientes()
        self.analise_administracao_indireta()

if __name__ == "__main__":
    df = pd.read_csv('dataset_preparado.csv', sep=';', encoding='utf-8')
    df['dataAdmissao'] = pd.to_datetime(df['dataAdmissao'])
    df['anosServico'] = (datetime.now() - df['dataAdmissao']).dt.days / 365.25
    analisador = AnalisadorServidores(df)
    analisador.gerar_relatorio_completo()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

class AnalisadorServidores:
    def __init__(self, df):
        self.df = df

    def analise_remuneracao(self):
        print("Remuneracao Media por Sexo:")
        remuneracao_sexo = self.df.groupby('sexo').agg({
            'valorBruto': ['mean', 'median', 'std'],
            'valorLiquido': ['mean', 'median'],
            'nomeServidor': 'count'
        }).round(2)
        print(remuneracao_sexo)

        print("Remuneracao Media por Tipo de Cargo (Top 10):")
        remuneracao_cargo = self.df.groupby('tipoCargo').agg({
            'valorBruto': ['mean', 'count']
        }).round(2).sort_values(('valorBruto', 'mean'), ascending=False).head(10)
        print(remuneracao_cargo)

        print("Remuneracao Media por Regime Contratual:")
        remuneracao_regime = self.df.groupby('regimeContratual').agg({
            'valorBruto': ['mean', 'median'],
            'nomeServidor': 'count'
        }).round(2)
        print(remuneracao_regime)

    def analise_descontos(self):
        print("Descontos Totais:")
        print(f"  Desconto medio: R$ {self.df['valorDesconto'].mean():.2f}")
        print(f"  Desconto mediano: R$ {self.df['valorDesconto'].median():.2f}")
        print(f"  Desconto maximo: R$ {self.df['valorDesconto'].max():.2f}")
        print(f"  Percentual de desconto medio: {self.df['descontosPercentual'].mean():.2f}%")

        print("Composicao dos Descontos (valores medios):")
        descontos_composicao = self.df[['valorPrevidenciario', 'valorIr', 'descontoObrigatorio']].mean()
        total_desconto_medio = self.df['valorDesconto'].mean()
        for coluna in descontos_composicao.index:
            percentual = (descontos_composicao[coluna] / total_desconto_medio * 100) if total_desconto_medio > 0 else 0
            print(f"  {coluna}: R$ {descontos_composicao[coluna]:.2f} ({percentual:.1f}%)")

        print("Descontos por Situacao do Servidor:")
        descontos_situacao = self.df.groupby('situacaoServidor').agg({
            'valorDesconto': ['mean', 'median'],
            'valorPrevidenciario': 'mean',
            'valorIr': 'mean'
        }).round(2)
        print(descontos_situacao)

    def analise_anos_servico(self):
        print("Estatisticas de Anos de Servico:")
        print(f"  Minimo: {self.df['anosServico'].min():.1f} anos")
        print(f"  Maximo: {self.df['anosServico'].max():.1f} anos")
        print(f"  Media: {self.df['anosServico'].mean():.1f} anos")
        print(f"  Mediana: {self.df['anosServico'].median():.1f} anos")

        self.df['categoriaAnosServico'] = pd.cut(
            self.df['anosServico'],
            bins=[0, 1, 5, 10, 20, float('inf')],
            labels=['< 1 ano', '1-5 anos', '5-10 anos', '10-20 anos', '> 20 anos']
        )

        print("Distribuicao por Categorias de Tempo de Servico:")
        dist_tempo = self.df['categoriaAnosServico'].value_counts().sort_index()
        for categoria, quantidade in dist_tempo.items():
            percentual = (quantidade / len(self.df)) * 100
            print(f"  {categoria}: {quantidade} ({percentual:.1f}%)")

        print("Correlacao: Anos de Servico vs Remuneracao:")
        correlacao = self.df['anosServico'].corr(self.df['valorBruto'])
        print(f"  Correlacao de Pearson: {correlacao:.3f}")

    def analise_distribuicao_orgaos(self):
        print("Top 10 Orgaos por Quantidade de Servidores:")
        orgaos = self.df['orgaoLotacao'].value_counts().head(10)
        for orgao, quantidade in orgaos.items():
            percentual = (quantidade / len(self.df)) * 100
            print(f"  {orgao}: {quantidade} ({percentual:.1f}%)")

        print("Top 10 Orgaos por Folha de Pagamento:")
        folha_por_orgao = self.df.groupby('orgaoLotacao')['valorBruto'].sum().sort_values(ascending=False).head(10)
        for orgao, folha in folha_por_orgao.items():
            print(f"  {orgao}: R$ {folha:,.2f}")

    def analise_escolaridade(self):
        print("Distribuicao por Escolaridade Minima:")
        escolaridade = self.df['escolaridadeMinimaCargo'].value_counts()
        for nivel, quantidade in escolaridade.items():
            percentual = (quantidade / len(self.df)) * 100
            print(f"  {nivel}: {quantidade} ({percentual:.1f}%)")

        print("Remuneracao Media por Escolaridade:")
        remuneracao_escolaridade = self.df.groupby('escolaridadeMinimaCargo').agg({
            'valorBruto': ['mean', 'median', 'count']
        }).round(2).sort_values(('valorBruto', 'mean'), ascending=False)
        print(remuneracao_escolaridade)

    def analise_pessoas_deficientes(self):
        total = len(self.df)
        com_deficiencia = (self.df['deficienteFisico'] == 'SIM').sum()
        sem_deficiencia = (self.df['deficienteFisico'] == 'NAO').sum()
        print(f"Total de Servidores: {total}")
        print(f"  Com deficiencia: {com_deficiencia} ({com_deficiencia/total*100:.2f}%)")
        print(f"  Sem deficiencia: {sem_deficiencia} ({sem_deficiencia/total*100:.2f}%)")
        if com_deficiencia > 0:
            print("Cargos de Pessoas com Deficiencia:")
            cargos_def = self.df[self.df['deficienteFisico'] == 'SIM']['nomeCargo'].value_counts().head(5)
            for cargo, quantidade in cargos_def.items():
                print(f"  {cargo}: {quantidade}")

    def analise_administracao_indireta(self):
        adm_indireta = self.df[self.df['administracao'] == 'ADMINISTRACAO INDIRETA']
        print(f"Total de Servidores em Admin. Indireta: {len(adm_indireta)}")
        print(f"   Percentual do total: {len(adm_indireta)/len(self.df)*100:.1f}%")
        print("Distribuicao por Entidade:")
        entidades = adm_indireta['cnpjOrgao'].value_counts()
        for entidade, quantidade in entidades.items():
            print(f"  {entidade}: {quantidade}")
        print("Folha de Pagamento:")
        print(f"  Total: R$ {adm_indireta['valorBruto'].sum():,.2f}")
        print(f"  Media por servidor: R$ {adm_indireta['valorBruto'].mean():,.2f}")

    def gerar_relatorio_completo(self):
        self.analise_remuneracao()
        self.analise_descontos()
        self.analise_anos_servico()
        self.analise_distribuicao_orgaos()
        self.analise_escolaridade()
        self.analise_pessoas_deficientes()
        self.analise_administracao_indireta()

if __name__ == "__main__":
    df = pd.read_csv('dataset_preparado.csv', sep=';', encoding='utf-8')
    df['dataAdmissao'] = pd.to_datetime(df['dataAdmissao'])
    df['anosServico'] = (datetime.now() - df['dataAdmissao']).dt.days / 365.25
    analisador = AnalisadorServidores(df)
    analisador.gerar_relatorio_completo()

In [ ]:
%pip install --upgrade nbformat

In [ ]:
import requests
import csv
import os

class SEADParaibaCSV:
    def __init__(self):
        self.base_url = "https://api.dadosabertos.codata.pb.gov.br/api/v1/remuneracao"
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/json",
            "Content-Type": "application/json"
        })

    def buscar_pagina(self, ano, mes, pagina):
        url = f"{self.base_url}/servidor"
        params = {
            "ano": str(ano),
            "mes": str(mes).zfill(2),
            "page": pagina
        }
        
        response = self.session.get(url, params=params)
        
        if response.status_code == 200:
            return response.json()
        else:
            print(f"Erro ao buscar página {pagina}. Status: {response.status_code}")
            return None

    def localizar_lista(self, dados_json):
        if isinstance(dados_json, list):
            return dados_json
            
        if isinstance(dados_json, dict):
            for chave, valor in dados_json.items():
                if isinstance(valor, list):
                    return valor
        return []

    def exportar_para_csv(self, ano, mes):
        nome_arquivo = f"servidores_pb_{ano}_{str(mes).zfill(2)}.csv"
        
        print(f"Iniciando extração. Os dados serão salvos em: {nome_arquivo}\n")

        arquivo_csv = None
        escritor = None
        cabecalhos = []
        pagina = 1

        while True:
            print(f"Baixando página {pagina}...")
            dados = self.buscar_pagina(ano, mes, pagina)
            
            if not dados:
                print("Falha na requisição. Encerrando busca.")
                break
                
            lista_servidores = self.localizar_lista(dados)
            
            if not lista_servidores:
                print(f"A lista de servidores na página {pagina} veio vazia. Fim da extração.")
                break
                
            for servidor in lista_servidores:
                if not cabecalhos:
                    cabecalhos = list(servidor.keys())
                    
                    arquivo_csv = open(nome_arquivo, mode="w", newline="", encoding="utf-8-sig")
                    escritor = csv.DictWriter(arquivo_csv, fieldnames=cabecalhos, delimiter=";")
                    escritor.writeheader()

                escritor.writerow(servidor)
            
            pagina += 1

        if arquivo_csv:
            arquivo_csv.close()

        total_paginas = pagina - 1
        print(f"\nExtração concluída com sucesso! Total de páginas processadas: {total_paginas}")
        print(f"Arquivo salvo no caminho: {os.path.abspath(nome_arquivo)}")

if __name__ == "__main__":
    extrator = SEADParaibaCSV()
    extrator.exportar_para_csv(ano=2026, mes=4)